# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAliEjaz1/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 (paper, Finding #4, "The Freshness Multiplier"):** pages older
than 365 days that were refreshed within the last 30 days show about 3.2x
higher health score (10.7 → 34.5) and about 57x more impressions (71 → 4,039)
than stale pages of the same age.

**My methodology question:** how were the refreshed and stale groups formed?
Refreshes are usually chosen by people, so pages picked for a refresh may
already have been the more promising ones. Also, the health score includes an
impressions component, so "health boost" and "impressions boost" overlap
partly by construction. The paper itself flags survivor bias in the
365+ / long-unchanged cell (active-content subset only). I'd ask whether the
same subset filter affects this refreshed-vs-stale comparison too.

**Finding 2 (paper, ML appendix, "What Predicts Growth?"):** a logistic
regression separates growing from declining pages with 71% holdout accuracy.

**My methodology question:** does the validation design carry that number?
The paper describes an 80/20 holdout but doesn't say whether the split is
random, by brand, or by time, and it doesn't state the majority-class
baseline. In Finding 1 about 62% of up/down pages are growing, so a model
that always says "growing" could score near 62% if the ML sample looks
similar. The label (30-day vs previous-30-day impression change) and features
like impressions and days visible also come from overlapping windows, so I'd
want to know how leakage between them was ruled out.

I ask these in the spirit of the paper's own standards: it labels the ML
section exploratory and discloses its limits. I'm practicing the next level of
rigor, not grading it.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_questions = {
    "finding_4_freshness": "how were refreshed vs stale groups formed; does health overlap with impressions by construction?",
    "ml_growth_model": "is the 71% accuracy above the majority-class baseline; how was the 80/20 split made; label/feature window overlap?",
}
for k, v in paper_questions.items():
    print(f"{k}: {v}")

finding_4_freshness: how were refreshed vs stale groups formed; does health overlap with impressions by construction?
ml_growth_model: is the 71% accuracy above the majority-class baseline; how was the 80/20 split made; label/feature window overlap?


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** the Week 5 design. Train on January→February, test on April→May,
but the same clients appear on both sides.

**After:** a grouped-by-client split on top of the time-aware split. 30% of
clients are held out entirely: none of their pages appear in training. Both
runs are scored on the same held-out-client test rows, and the Week 4
baseline is scored on those rows too. I also report the base rate and lift,
because precision means little without knowing how common declines are.


Note: the table above is one draw of 15 held-out clients and turned out to be
an unfavorable one. Section 3 repeats it over 10 draws.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"
FEATURES = ["avg_position", "total_impressions", "ctr"]

def load_month_features(month):
    path = f"{BASE}/fact_content_daily_performance/month={month}/data_0.parquet"
    df = con.execute(f"""
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_avg_position) AS avg_position,
               SUM(gsc_impressions) AS total_impressions,
               SUM(gsc_clicks) AS total_clicks
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    """).df()
    df["ctr"] = np.where(df["total_impressions"] > 0,
                         df["total_clicks"] / df["total_impressions"], 0.0)
    return df

def build_pair(month_a, month_b, min_impressions=100):
    a = load_month_features(month_a)
    b = load_month_features(month_b)[["content_hash_id", "client_hash_id", "total_clicks"]]
    b = b.rename(columns={"total_clicks": "clicks_next"})
    m = a.merge(b, on=["content_hash_id", "client_hash_id"], how="inner")
    m = m[m["total_impressions"] >= min_impressions].copy()
    m["is_declining"] = (m["clicks_next"] < 0.8 * m["total_clicks"]).astype(int)
    return m

def baseline_score_row(r):
    if 11 <= r["avg_position"] <= 20 and r["total_impressions"] >= 100:
        return 400 - r["avg_position"]
    elif r["avg_position"] <= 10 and r["ctr"] < 0.01 and r["total_impressions"] >= 100:
        return 200 + min(r["total_impressions"] / 1000, 99)
    return min(r["total_impressions"] / 10000, 99)

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(labels)[order].mean())

def evaluate(train, test, k=50):
    tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
    tree.fit(train[FEATURES], train["is_declining"])
    model_s = tree.predict_proba(test[FEATURES])[:, 1]
    base_s = test.apply(baseline_score_row, axis=1).values
    base_rate = test["is_declining"].mean()
    tree_p = precision_at_k(model_s, test["is_declining"], k)
    rule_p = precision_at_k(base_s, test["is_declining"], k)
    return {"n_train": len(train), "n_test": len(test),
            "base_rate": round(base_rate, 3),
            f"baseline_P@{k}": round(rule_p, 3),
            f"tree_P@{k}": round(tree_p, 3),
            "tree_lift": round(tree_p / base_rate, 2),
            "distinct_tree_scores": len(np.unique(model_s))}

train_df = build_pair("2026-01", "2026-02")
test_df = build_pair("2026-04", "2026-05")

# hold out 30% of clients entirely
rng = np.random.RandomState(42)
clients = np.array(sorted(set(train_df["client_hash_id"]) | set(test_df["client_hash_id"])))
held_out = set(rng.permutation(clients)[: int(0.3 * len(clients))])

train_g = train_df[~train_df["client_hash_id"].isin(held_out)]
test_g = test_df[test_df["client_hash_id"].isin(held_out)]
assert not (set(train_g["client_hash_id"]) & set(test_g["client_hash_id"]))

res_week5 = evaluate(train_df, test_df)   # original Week 5 design
res_before = evaluate(train_df, test_g)   # same test rows as "after", clients seen in training
res_after = evaluate(train_g, test_g)     # test clients never seen in training

table = pd.DataFrame({"Week 5 design (all clients)": res_week5,
                      "Before (held-out test rows, clients seen)": res_before,
                      "After (grouped by client)": res_after}).T
print(f"Clients total: {len(clients)}, held out: {len(held_out)}")
table

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients total: 50, held out: 15


,n_train,n_test,base_rate,baseline_P@50,tree_P@50,tree_lift,distinct_tree_scores
Week 5 design (all clients),63831.0,106586.0,0.270,0.34,0.74,2.74,12.0
"Before (held-out test rows, clients seen)",63831.0,29357.0,0.247,0.38,0.62,2.51,11.0
After (grouped by client),46845.0,29357.0,0.247,0.38,0.26,1.05,11.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Audit of the final feature set, including the label definition itself.

1. **Label artifact:** the label is `clicks_next < 0.8 * total_clicks`. A page
   with zero clicks in the feature month can never satisfy it, so every
   zero-click page is "not declining" by construction. I count them and
   re-run on eligible pages only (clicks > 0).
2. **Overlap:** the same pages appear in both the train and test months. I
   count them.
3. **Feature timing:** all three features come from the feature month only.
4. **Stability:** the grouped result above is one random draw of 15 held-out
   clients, so I repeat it over 10 different draws.
5. **Ties:** a depth-4 tree gives few distinct scores, so the top 50 contains
   arbitrary ties.

  **Findings:** 39.6% of feature-month pages had zero clicks and cannot be
labeled declining by construction (decline rate 0.000). Across 10 different
held-out-client draws, the tree averaged P@50 of 0.60 vs 0.30 for the Week 4
rule and won in 9 of 10 draws, but ranged from 0.26 to 0.80. On eligible
pages only (clicks > 0, one draw) the tree scored 0.64 vs 0.44 for the
baseline, with a base rate of 0.40.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Label artifact: zero-click pages can never be labeled "declining"
zero = train_df[train_df["total_clicks"] == 0]
print(f"Zero-click pages in feature month: {len(zero)} of {len(train_df)} "
      f"({len(zero)/len(train_df):.1%}); decline rate among them: {zero['is_declining'].mean():.3f}")

# 2. Overlap between train and test months
shared_pages = set(train_df["content_hash_id"]) & set(test_df["content_hash_id"])
print(f"Pages appearing in both train and test months: {len(shared_pages)}")

# 3. Feature timing table
audit = pd.DataFrame({
    "feature": FEATURES,
    "source_month": ["feature month only"] * 3,
    "knowable_at_decision_time": ["yes: same-month Search Console aggregate"] * 3,
})
print(audit.to_string(index=False))

# 4. Repeat the grouped split with 10 different client draws
rows = []
for seed in range(10):
    rng_s = np.random.RandomState(seed)
    ho = set(rng_s.permutation(clients)[: int(0.3 * len(clients))])
    tr = train_df[~train_df["client_hash_id"].isin(ho)]
    te = test_df[test_df["client_hash_id"].isin(ho)]
    r = evaluate(tr, te)
    rows.append({"seed": seed, "base_rate": r["base_rate"],
                 "baseline_P@50": r["baseline_P@50"], "tree_P@50": r["tree_P@50"],
                 "tree_lift": r["tree_lift"]})
repeat = pd.DataFrame(rows)
print(repeat.to_string(index=False))
print(repeat[["baseline_P@50", "tree_P@50", "tree_lift"]].agg(["mean", "min", "max"]).round(3))

# 5. Eligible pages only (clicks > 0 in feature month), same held-out clients as Section 2
tr_e = train_g[train_g["total_clicks"] > 0]
te_e = test_g[test_g["total_clicks"] > 0]
res_eligible = evaluate(tr_e, te_e)
pd.DataFrame({"After (grouped, all pages)": res_after,
              "Grouped, eligible pages only (clicks>0)": res_eligible}).T

Zero-click pages in feature month: 25259 of 63831 (39.6%); decline rate among them: 0.000
Pages appearing in both train and test months: 52792
          feature       source_month                knowable_at_decision_time
     avg_position feature month only yes: same-month Search Console aggregate
total_impressions feature month only yes: same-month Search Console aggregate
              ctr feature month only yes: same-month Search Console aggregate
 seed  base_rate  baseline_P@50  tree_P@50  tree_lift
    0      0.287           0.26       0.50       1.74
    1      0.278           0.24       0.68       2.45
    2      0.243           0.32       0.32       1.32
    3      0.248           0.36       0.66       2.66
    4      0.251           0.26       0.28       1.12
    5      0.294           0.26       0.70       2.38
    6      0.250           0.28       0.62       2.48
    7      0.272           0.26       0.72       2.64
    8      0.257           0.36       0.74       2.88
    9

,n_train,n_test,base_rate,baseline_P@50,tree_P@50,tree_lift,distinct_tree_scores
"After (grouped, all pages)",46845.0,29357.0,0.247,0.38,0.26,1.05,11.0
"Grouped, eligible pages only (clicks>0)",27875.0,18149.0,0.400,0.44,0.78,1.95,15.0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (Week 5):** "The tree beats the baseline: 72% of the top 50
flagged pages are genuinely declining."

**Problems found in this audit:**
- Test clients were also seen in training. Grouping by client fixes this.
- Zero-click pages (39.6%) are labeled non-declining by construction, which
  inflates results unless removed.
- "Genuinely declining" over-claims. The label measures a 20%+ month-over-month
  click drop, which can also come from seasonality or noise.
- A single split hides variance: results ranged from 0.26 to 0.80 across
  client draws.
- The tree gives only 11-15 distinct scores, so the top 50 contains arbitrary
  ties.

**Rewritten claim:** see the printed sentence below.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
wins = int((repeat["tree_P@50"] > repeat["baseline_P@50"]).sum())
print(
    f"Measured on held-out clients (April features, May outcome), the decision tree's "
    f"top-50 queue averaged {repeat['tree_P@50'].mean():.0%} pages with an observed click drop "
    f"of 20%+, versus {repeat['baseline_P@50'].mean():.0%} for the Week 4 rule, across "
    f"{len(repeat)} random client draws (base rate about {repeat['base_rate'].mean():.0%}). "
    f"The tree scored higher in {wins} of {len(repeat)} draws, but its score ranged from "
    f"{repeat['tree_P@50'].min():.0%} to {repeat['tree_P@50'].max():.0%}, so the ordering is "
    f"directional, not stable. On pages with clicks in the feature month, one draw gave "
    f"{res_eligible['tree_P@50']:.0%} vs {res_eligible['baseline_P@50']:.0%} for the baseline. "
    f"This is decision-support evidence for prioritizing review across 50 clients. It does "
    f"not show that any signal causes decline or that refreshing a page recovers clicks."
)

Measured on held-out clients (April features, May outcome), the decision tree's top-50 queue averaged 60% pages with an observed click drop of 20%+, versus 30% for the Week 4 rule, across 10 random client draws (base rate about 27%). The tree scored higher in 9 of 10 draws, but its score ranged from 28% to 80%, so the ordering is directional, not stable. On pages with clicks in the feature month, one draw gave 78% vs 44% for the baseline. This is decision-support evidence for prioritizing review across 50 clients. It does not show that any signal causes decline or that refreshing a page recovers clicks.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.